# Lab 23 — Train a Small Language Model from Scratch

Lab 16 built the transformer architecture. This lab builds the **entire training pipeline around it** — real tokenizer, real corpus, LR schedule, validation loop, checkpointing, perplexity tracking, sampling. Everything you'd find in a production training run at 100–1000× this scale.

We're training on **[TinyStories](https://huggingface.co/datasets/roneneldan/TinyStories)** (Eldan & Li, 2023 — [arXiv:2305.07759](https://arxiv.org/abs/2305.07759)). The dataset is a breakthrough contribution: ~2M short English stories at a 3–4-year-old's vocabulary level, synthetically generated by GPT-3.5/4. Eldan and Li showed that with this data you can train a **10–30M param model that produces coherent, grammatical, internally-consistent stories** — debunking the idea that you need a frontier-scale model for coherent text. The paper kicked off an entire wave of research into "SLM" (Small Language Model) training.

### The key insight

At very small scales, model quality is **almost entirely** bottlenecked by the *difficulty* of the training data. On web-crawled text, a 10M-param model produces gibberish. On TinyStories (curated simple data), the same 10M-param model produces correct children's stories. Data > parameters at this scale.

This maps to Microsoft's Phi-3 strategy at larger scale: **filter/generate high-quality data → train a much smaller model to frontier quality**.

### References to read while training runs

- **[TinyStories (Eldan & Li, 2023)](https://arxiv.org/abs/2305.07759)** — the paper behind our corpus.
- **[Chinchilla (Hoffmann et al., 2022)](https://arxiv.org/abs/2203.15556)** — the scaling law paper. Models should be trained with ~20× more tokens than parameters. Our 2M-param model ↔ ~40M tokens of data (we have ~3M for speed, so we're under-trained).
- **[Karpathy's nanoGPT](https://github.com/karpathy/nanoGPT)** — the canonical readable reference training loop. Our loop tracks its structure.
- **[Andrej Karpathy — Let's reproduce GPT-2 (124M)](https://www.youtube.com/watch?v=l8pRSuU81PU)** — 4-hour video reproducing GPT-2 from scratch including all the optimizer tricks.

### What you'll build

1. **Data pipeline** — stream TinyStories, tokenize with GPT-2 BPE, pack into efficient training tensors, held-out validation split
2. **Model + optimizer** — compact GPT (~1.5M params), AdamW, linear warmup → cosine decay LR schedule
3. **Training loop** — loss tracking, periodic validation, perplexity reporting
4. **Generation + checkpointing** — before/after samples, save + reload the model

Runtime target on RTX 3060 Ti: ~2 min.

---

## Step 1 — Data pipeline

Production training does three things in the data layer and we'll mirror all three:

1. **Real tokenizer.** Char-level toys (Lab 16) train fast but produce terrible text. Real LMs use BPE — we'll reuse GPT-2's pretrained tokenizer (50,257 vocab). Same tokenizer used by GPT-3, Codex, many Llama variants.
2. **Packed concatenation.** Rather than per-story padding, real training concatenates all documents with an EOS token between them and chops into fixed-length sequences. Much less padding waste, ~2× throughput.
3. **Train/val split.** Held-out tokens for tracking generalization. ~1% of the corpus is plenty for perplexity tracking.

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from datasets import load_dataset
from transformers import AutoTokenizer
import time
import math

device = torch.device('cuda')
torch.manual_seed(1337)

# 1. Load a slice of TinyStories. Full set is ~2M stories (~4GB tokenized);
#    we take the first 5000 for fast iteration. Stream to avoid the full download.
print('Loading TinyStories (streaming first 5000 entries)...')
ds = load_dataset('roneneldan/TinyStories', split='train')
stories = []
for i, ex in enumerate(ds):
    if i >= 5000:
        break
    stories.append(ex['text'])

total_chars = sum(len(s) for s in stories)
print(f'Loaded {len(stories):,} stories, {total_chars:,} chars total')

/usr/local/lib/python3.11/dist-packages/torch/cuda/__init__.py:65: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


Loading TinyStories (streaming first 5000 entries)...


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00004-2d5a1467fff108(…):   0%|          | 0.00/249M [00:00<?, ?B/s]

data/train-00001-of-00004-5852b56a2bd28f(…):   0%|          | 0.00/248M [00:00<?, ?B/s]

...

data/train-00003-of-00004-d243063613e5a0(…):   0%|          | 0.00/248M [00:00<?, ?B/s]

data/validation-00000-of-00001-869c898b5(…):   0%|          | 0.00/9.99M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/2119719 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/21990 [00:00<?, ? examples/s]

Loaded 5,000 stories, 4,176,740 chars total


In [5]:
stories[:2]

['One day, a little girl named Lily found a needle in her room. She knew it was difficult to play with it because it was sharp. Lily wanted to share the needle with her mom, so she could sew a button on her shirt.\n\nLily went to her mom and said, "Mom, I found this needle. Can you share it with me and sew my shirt?" Her mom smiled and said, "Yes, Lily, we can share the needle and fix your shirt."\n\nTogether, they shared the needle and sewed the button on Lily\'s shirt. It was not difficult for them because they were sharing and helping each other. After they finished, Lily thanked her mom for sharing the needle and fixing her shirt. They both felt happy because they had shared and worked together.',
 'Once upon a time, there was a little car named Beep. Beep loved to go fast and play in the sun. Beep was a healthy car because he always had good fuel. Good fuel made Beep happy and strong.\n\nOne day, Beep was driving in the park when he saw a big tree. The tree had many leaves that we

In [2]:
# 2. Tokenize + pack into contiguous token streams
tokenizer = AutoTokenizer.from_pretrained('gpt2')
EOS = tokenizer.eos_token_id
print(f'Vocab: {tokenizer.vocab_size:,}, EOS token id: {EOS}')

# Tokenize each story, concatenate with EOS between them
print('Tokenizing...')
all_ids = []
for s in stories:
    all_ids.extend(tokenizer.encode(s) + [EOS])

all_tokens = torch.tensor(all_ids, dtype=torch.long)

# Held-out val split — last ~1% of the corpus
split = int(0.99 * len(all_tokens))
train_tokens = all_tokens[:split].to(device)
val_tokens = all_tokens[split:].to(device)

print(f'Train: {len(train_tokens):,} tokens')
print(f'Val:   {len(val_tokens):,} tokens')
print(f'Total tokens: {len(all_tokens):,} (vs 20 * params rule-of-thumb for a compute-optimal model)')

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Vocab: 50,257, EOS token id: 50256
Tokenizing...


Token indices sequence length is longer than the specified maximum sequence length for this model (1106 > 1024). Running this sequence through the model will result in indexing errors


Train: 1,022,756 tokens
Val:   10,331 tokens
Total tokens: 1,033,087 (vs 20 * params rule-of-thumb for a compute-optimal model)


In [6]:
from preporato_labs import Lab
lab = Lab('train-slm')
lab.check(1)

OK — tokenized: train=1,022,756 tokens, val=10,331 tokens, vocab_size=50,257
STEP_PASSED


Step 1 Complete! Scroll down to continue...

True

## Step 2 — Model + optimizer + LR schedule

We reuse the transformer from Lab 16 essentially verbatim, sized to ~1.5M params. At this size, training on ~3M tokens takes about 2 minutes.

### The learning rate schedule (the single most impactful hyperparam you've never tuned)

Flat LR = slow + unstable. Real training uses:

- **Warmup**: the first N (e.g. 100) steps linearly ramp LR from 0 to peak. Stops early-training divergence.
- **Cosine decay**: after warmup, LR smoothly decays from peak down to ~10% of peak by end of training.

This schedule is used by virtually every published LLM trained since 2019 (Chinchilla, Llama, GPT-3, Mistral). Deviating from it almost always hurts.

### 🐛 Common mistake: forgetting weight decay

AdamW without weight decay drifts into overfitting and produces worse gradients later in training. Llama, GPT-3, all modern LLMs use weight decay = 0.1 on most weights. Critical detail: **biases and LayerNorm parameters should NOT be decayed** — that's a canonical footgun.

### Multi-head attention — the compute-dense core

The attention block projects input into query/key/value heads, runs scaled-dot-product attention (PyTorch's built-in dispatches to FlashAttention when available), and projects back. `is_causal=True` gives us the autoregressive mask for free — no manual triangular mask needed.

In [11]:
class MHA(nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        self.qkv = nn.Linear(d_model, 3 * d_model, bias=False)
        self.proj = nn.Linear(d_model, d_model, bias=False)

    def forward(self, x):
        B, T, D = x.shape
        q, k, v = self.qkv(x).split(D, dim=-1)
        q = q.view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        k = k.view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        v = v.view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        # Use PyTorch's built-in scaled_dot_product_attention (FlashAttention under the hood when supported)
        out = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        return self.proj(out.transpose(1, 2).contiguous().view(B, T, D))

### Transformer block — pre-LN + residual

Pre-LayerNorm (GPT-2 style, `ln1(x)` before attention) is more stable to train than the original post-LN. FFN is the usual 4× hidden-size expansion with GELU. Residual connections make deep stacks trainable.

In [13]:
class Block(nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = MHA(d_model, n_heads)
        self.ln2 = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(nn.Linear(d_model, 4*d_model), nn.GELU(), nn.Linear(4*d_model, d_model))

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.ffn(self.ln2(x))
        return x

### SmallGPT — the whole model, assembled

Token + position embeddings, N transformer blocks, final LayerNorm, output projection. Two production tricks baked in:

1. **Weight tying** — the output projection shares weights with `tok_emb`. Saves a vocab×d params (significant on small models) and often improves perplexity.
2. **GPT-2 init (std=0.02)** — PyTorch's default Embedding init has std=1.0, which combined with weight tying gives gigantic initial logits and a loss spike at step 0. std=0.02 keeps the initial loss near `log(vocab_size)`, a sanity-check every GPT trainer looks for.

In [14]:
class SmallGPT(nn.Module):
    def __init__(self, vocab_size, d_model=128, n_heads=4, n_layers=4, max_len=128):
        super().__init__()
        self.max_len = max_len
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(max_len, d_model)
        self.blocks = nn.ModuleList([Block(d_model, n_heads) for _ in range(n_layers)])
        self.ln_f = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size, bias=False)
        # Weight tying: reuse the embedding matrix as the output projection (saves params, often improves quality)
        self.head.weight = self.tok_emb.weight
        # GPT-2 style init: default nn.Embedding std is 1.0 which combined with weight tying
        # produces enormous initial logits. std=0.02 keeps initial loss near log(vocab_size).
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        x = self.tok_emb(idx) + self.pos_emb(torch.arange(T, device=idx.device))
        for blk in self.blocks:
            x = blk(x)
        x = self.ln_f(x)
        logits = self.head(x)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss

model = SmallGPT(vocab_size=tokenizer.vocab_size, d_model=128, n_heads=4, n_layers=4, max_len=128).to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f'Model: {n_params/1e6:.2f}M params')
print(f'VRAM used (weights in fp32): ~{n_params * 4 / 1e6:.1f} MB')

Model: 7.24M params
VRAM used (weights in fp32): ~29.0 MB


In [15]:
# Split parameters into decayed vs undecayed groups
decay_params, no_decay_params = [], []
for name, p in model.named_parameters():
    if p.dim() >= 2:          # matrices → apply weight decay
        decay_params.append(p)
    else:                      # biases, LayerNorm scales/biases → no decay
        no_decay_params.append(p)

PEAK_LR = 3e-3
optimizer = torch.optim.AdamW([
    {'params': decay_params, 'weight_decay': 0.1},
    {'params': no_decay_params, 'weight_decay': 0.0},
], lr=PEAK_LR, betas=(0.9, 0.95), fused=True)

# Warmup + cosine decay
TOTAL_STEPS = 400
WARMUP = 40
def lr_schedule(step):
    if step < WARMUP:
        return step / WARMUP
    # Cosine from 1.0 down to 0.1
    progress = (step - WARMUP) / (TOTAL_STEPS - WARMUP)
    return 0.1 + 0.9 * 0.5 * (1 + math.cos(math.pi * progress))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_schedule)
print(f'LR schedule: warmup {WARMUP} steps -> peak {PEAK_LR} -> cosine decay to {0.1*PEAK_LR} over {TOTAL_STEPS} steps')

LR schedule: warmup 40 steps -> peak 0.003 -> cosine decay to 0.00030000000000000003 over 400 steps


In [16]:
lab.check(2)

OK — model: 7.24M params on cuda:0, optimizer + LR schedule ready
STEP_PASSED


Step 2 Complete! Scroll down to continue...

True

## Step 3 — Training loop with validation + perplexity

Two things the loop does differently from Lab 16's toy version:

- **Periodic validation**. Every K steps we evaluate on the held-out val set. If val loss stops improving, we're either done or overfitting.
- **Perplexity report**. Val loss is a number; val perplexity = exp(val_loss) is the interpretable number — the "effective branching factor" we learned about in Lab 8. You want perplexity on held-out data to keep falling through training.

At this scale, you should see:
- Initial train loss around 10 (random ~ log(50257) ≈ 10.8)
- Final train loss ~3.5-4.5
- Final val perplexity ~40-80 (a fully-trained production SLM on TinyStories gets ~5-15; we're running 50-100x less compute)

### Batch sampler + validation estimator

`get_batch` samples `BATCH_SIZE` random starting positions in the token stream and returns (x, y) pairs where y is x shifted by one — the next-token prediction target. `estimate_val_loss` runs the model on 10 val batches to get a stable loss estimate without full-dataset overhead.

In [17]:
BLOCK_SIZE = 128
BATCH_SIZE = 32

def get_batch(split='train'):
    data = train_tokens if split == 'train' else val_tokens
    # Sample BATCH_SIZE starting positions, each producing a BLOCK_SIZE-long sequence
    ix = torch.randint(0, len(data) - BLOCK_SIZE - 1, (BATCH_SIZE,), device=device)
    x = torch.stack([data[i:i+BLOCK_SIZE] for i in ix])
    y = torch.stack([data[i+1:i+1+BLOCK_SIZE] for i in ix])
    return x, y

@torch.no_grad()
def estimate_val_loss(n_batches=10):
    model.eval()
    losses = []
    for _ in range(n_batches):
        x, y = get_batch('val')
        _, loss = model(x, y)
        losses.append(loss.item())
    model.train()
    return sum(losses) / len(losses)

### Training loop — gradient-clipped, LR-scheduled

Standard GPT training loop with two production defaults every implementation should have:

1. **`grad_norm_ <= 1.0`** — clipping prevents the occasional giant gradient (common with character-level LM, attention collapses, bad batches) from blowing up the optimizer state.
2. **Scheduler step per-step, not per-epoch** — linear warmup + cosine decay pacing. Reading `scheduler.get_last_lr()[0]` in the log line is invaluable when tuning learning rates.

Val is sampled every 50 steps: frequent enough to see the curve bend but cheap enough not to dominate wall clock.

In [18]:
# Training loop
train_losses = []
val_losses = []
val_steps = []

model.train()
start = time.time()
for step in range(TOTAL_STEPS):
    x, y = get_batch('train')
    _, loss = model(x, y)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()
    scheduler.step()
    train_losses.append(loss.item())

    if step % 50 == 0 or step == TOTAL_STEPS - 1:
        v = estimate_val_loss()
        val_losses.append(v)
        val_steps.append(step)
        elapsed = time.time() - start
        print(f'step {step:4d}  train {loss.item():.3f}  val {v:.3f}  val_ppl {math.exp(v):6.1f}  lr {scheduler.get_last_lr()[0]:.2e}  [{elapsed:.1f}s]')

print(f'\nTraining took {time.time() - start:.1f}s over {TOTAL_STEPS} steps')
print(f'Final val perplexity: {math.exp(val_losses[-1]):.2f}')

step    0  train 10.854  val 10.853  val_ppl 51680.1  lr 7.50e-05  [0.2s]
step   50  train 5.470  val 5.581  val_ppl  265.2  lr 2.99e-03  [0.9s]
step  100  train 4.382  val 4.576  val_ppl   97.2  lr 2.81e-03  [1.7s]
step  150  train 4.097  val 4.268  val_ppl   71.4  lr 2.41e-03  [2.4s]
step  200  train 3.944  val 4.068  val_ppl   58.4  lr 1.87e-03  [3.1s]
step  250  train 3.751  val 3.916  val_ppl   50.2  lr 1.29e-03  [3.8s]
step  300  train 3.745  val 3.853  val_ppl   47.1  lr 7.73e-04  [4.6s]
step  350  train 3.663  val 3.757  val_ppl   42.8  lr 4.22e-04  [5.3s]
step  399  train 3.619  val 3.717  val_ppl   41.2  lr 3.00e-04  [6.0s]

Training took 6.0s over 400 steps
Final val perplexity: 41.15


In [19]:
lab.check(3)

OK — train loss 10.771 -> 3.581 (67% reduction); val perplexity = 41.15
STEP_PASSED


Step 3 Complete! Scroll down to continue...

True

## Step 4 — Generate + checkpoint

Two last production-critical capabilities:

- **Sampling**: generation with temperature and top-k filtering, to avoid the greedy repetition we'd get from `argmax`
- **Checkpointing**: save the model's state dict so we can resume or deploy. We'll do a save + reload round-trip to verify it works.

We'll also generate from the **untrained** model for direct comparison. Untrained = random output over the 50K vocab = word-salad. Trained = recognisable English story continuation.

In [21]:
@torch.no_grad()
def generate(model, prompt, max_new=80, temperature=0.8, top_k=40):
    model.eval()
    ids = torch.tensor([tokenizer.encode(prompt)], dtype=torch.long, device=device)
    for _ in range(max_new):
        # Crop to model's max context
        idx_cond = ids[:, -model.max_len:]
        logits, _ = model(idx_cond)
        logits = logits[:, -1, :] / temperature
        # Top-k filter: keep only the k most likely tokens, zero out the rest
        if top_k is not None:
            v, _ = torch.topk(logits, top_k)
            logits[logits < v[:, [-1]]] = -float('inf')
        probs = F.softmax(logits, dim=-1)
        next_id = torch.multinomial(probs, num_samples=1)
        ids = torch.cat([ids, next_id], dim=1)
    return tokenizer.decode(ids[0].tolist())

prompt = 'Once upon a time, there was'

# Untrained baseline for comparison (we need a fresh model — our `model` is already trained)
fresh = SmallGPT(vocab_size=tokenizer.vocab_size, d_model=128, n_heads=4, n_layers=4, max_len=128).to(device)
story_before = generate(fresh, prompt, max_new=60, temperature=1.0, top_k=None)
del fresh

story_after = generate(model, prompt, max_new=60)

print('UNTRAINED (random weights):')
print(f'  {story_before!r}')
print()
print('TRAINED:')
print(f'  {story_after!r}')

UNTRAINED (random weights):
  'Once upon a time, there was brighterHongem genesisLightAMI ALSO quiet Ducksuringplug designate breathe vandalism molecular � FilePlayer130clusivelysoType two star Were "[ occupancyjar translthoughfalls FactorsAzellectualCapital Divideaked Zi intrinsicomical Hamm Kro240 Aadactic � vaccines Titanic Syndicatetexture Rebeludder graduate metroVanpkg bout mesh Ply Sport Pricing'

TRAINED:
  'Once upon a time, there was a brave little girl named Mommy. "Wow, little boy, Timmy! I have to me and we can have been fun."\n\nHer mommy replied if it was time and said, â€œI\'m you need to a good for you can I have to worry'


In [22]:
# Checkpoint round-trip — save, clear, reload, verify output matches
CKPT = '/tmp/slm_checkpoint.pt'
torch.save({
    'model_state_dict': model.state_dict(),
    'config': {'vocab_size': tokenizer.vocab_size, 'd_model': 128, 'n_heads': 4, 'n_layers': 4, 'max_len': 128},
    'train_step': TOTAL_STEPS,
    'final_val_loss': val_losses[-1],
}, CKPT)
print(f'Saved to {CKPT}')

# Load into a fresh instance and confirm identical output
ckpt = torch.load(CKPT, weights_only=True)
cfg = ckpt['config']
reloaded = SmallGPT(**cfg).to(device)
reloaded.load_state_dict(ckpt['model_state_dict'])
reloaded.eval()

# Deterministic sanity check — same seed, same output
torch.manual_seed(42)
ids = torch.tensor([tokenizer.encode('The')], dtype=torch.long, device=device)
orig_logits, _ = model(ids)
reload_logits, _ = reloaded(ids)
checkpoint_ok = torch.allclose(orig_logits, reload_logits, atol=1e-5)
print(f'Checkpoint round-trip: {"OK" if checkpoint_ok else "MISMATCH!"} (max abs diff: {(orig_logits-reload_logits).abs().max().item():.2e})')

Saved to /tmp/slm_checkpoint.pt
Checkpoint round-trip: OK (max abs diff: 0.00e+00)


In [23]:
lab.check(4)

OK — trained story: 48 words, 97.9% alphabetic; checkpoint round-trips cleanly
STEP_PASSED


Step 4 Complete! Lab complete!

True

---

## What you just built

A full production-shaped SLM training pipeline on ~1.5M params — data streaming, BPE tokenization, packed sequences, validation loop, LR warmup + cosine decay, weight decay on the right params, grad clipping, checkpointing. Every piece scales unchanged to 8B param runs; only the config numbers change.

## What to read next

- **[TinyStories paper (Eldan & Li, 2023)](https://arxiv.org/abs/2305.07759)** — why data quality dominates at small scales.
- **[Chinchilla paper (Hoffmann et al., 2022)](https://arxiv.org/abs/2203.15556)** — scaling laws: model size vs data size vs compute.
- **[Karpathy — Let's reproduce GPT-2 (124M)](https://www.youtube.com/watch?v=l8pRSuU81PU)** — 4-hour deep dive reproducing GPT-2 including all tricks (flash attn, gradient checkpointing, DDP, etc).
- **[MAP-Neo (2024)](https://arxiv.org/abs/2405.19327)** — a fully open 7B training with full data and code. Canonical for "train a real LM from nothing".
- **[μP (maximal update parameterization)](https://arxiv.org/abs/2203.03466)** — LR-free scaling. If you plan to tune hyperparams at tiny scale and transfer to large, read this.

## What to try next

- Scale `n_layers` from 4 → 8 and `d_model` from 128 → 256 (~15M params). Does val perplexity drop? How much longer does training take?
- Replace the LR schedule with a flat LR and observe the difference in final val loss — usually a ~20–40% degradation.
- Train for 10× more steps and watch for overfitting onset in the val loss.
- Swap TinyStories for [OpenWebText](https://huggingface.co/datasets/Skylion007/openwebtext) — real web text is dramatically harder and exposes the small-model ceiling.